# Notebook 05 — UMAP Projection and HDBSCAN Clustering

This notebook:
1. Loads embeddings from the cache
2. Projects them to 2D with UMAP
3. Clusters with HDBSCAN
4. Saves the results to `data/embeddings/projections.json` and `clusters.json`

**Before running:** complete notebook 04 to generate embeddings.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from pathlib import Path
from src.utils.config import load_config
from src.embed.embeddings import embeddings_to_matrix, _load_cache
from src.cluster.cluster import reduce_dimensions, cluster_hdbscan, attach_results_to_df, save_projections

cfg = load_config()

In [ ]:
# --- 1. Load all artifacts and embedding cache ---

dfs = []
for name in ['papers', 'patents', 'projects', 'parts']:
    path = Path(f'../data/processed/{name}.csv')
    if path.exists():
        dfs.append(pd.read_csv(path))

combined = pd.concat(dfs, ignore_index=True)
cache = _load_cache('../data/embeddings/embeddings.json')
print(f'{len(combined)} artifacts, {len(cache)} cached embeddings')

In [ ]:
# --- 2. Build the embedding matrix ---

import numpy as np
from src.embed.embeddings import embeddings_to_matrix

matrix, valid_ids = embeddings_to_matrix(combined, cache)
print(f'Embedding matrix: {matrix.shape}')

In [ ]:
# --- 3. UMAP ---

umap_cfg = cfg['umap']
coords_2d = reduce_dimensions(
    matrix,
    n_components=umap_cfg['n_components'],
    n_neighbors=umap_cfg['n_neighbors'],
    min_dist=umap_cfg['min_dist'],
    metric=umap_cfg['metric'],
    random_state=umap_cfg['random_state'],
)
print(f'UMAP output: {coords_2d.shape}')

In [ ]:
# --- 4. HDBSCAN ---

cl_cfg = cfg['clustering']
labels = cluster_hdbscan(
    coords_2d,
    min_cluster_size=cl_cfg['min_cluster_size'],
    min_samples=cl_cfg['min_samples'],
)

import pandas as pd
print(pd.Series(labels).value_counts().sort_index().head(20))

In [ ]:
# --- 5. Save results ---

save_projections(valid_ids, coords_2d, labels, '../data/embeddings/projections.json')

combined_with_clusters = attach_results_to_df(combined, valid_ids, coords_2d, labels)
combined_with_clusters.to_csv('../data/processed/all_artifacts.csv', index=False)
print('Saved projections.json and all_artifacts.csv')